# 07 · Risco de progressão entre grupos

Para cada país, estimamos a **probabilidade de avançar para um grupo de fecundidade mais baixo** no próximo intervalo de tempo. "Avançar" significa mover-se da fase de transição para uma fase posterior (por exemplo, de transição inicial para transição avançada).

**Método:** modelo logístico de tempo discreto. Em cada intervalo entre duas épocas, o desfecho é 1 se o país avançou e 0 se não avançou. As covariáveis são a fecundidade e a expectativa de vida no início do intervalo, além do grupo de partida.

**Dados:** grupos por época do notebook 04 (`grupos_por_epoca.csv`). Os grupos são numerados do mais baixo (0, baixa fecundidade) ao mais alto (3, alta fecundidade). Como cada época é agrupada separadamente, o número de um grupo pode não corresponder exatamente à mesma fase em épocas diferentes; o nome do grupo é dado pela fecundidade média da época.

**Limitações:** as épocas de 2030 em diante usam projeções, e o modelo é estimado com poucos intervalos de 20 anos. As probabilidades devem ser lidas como estimativas relativas entre países, não como previsões precisas.

In [1]:
import sys
from pathlib import Path

# Permite importar src/ a partir da pasta notebooks/
if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import altair as alt
import numpy as np
import pandas as pd
from statsmodels.formula.api import logit
from vega_datasets import data

from src.dados import carregar_medium, paises

RAIZ = Path.cwd().parent
medium = carregar_medium()
paises_medium = paises(medium)
MALHA = alt.topo_feature(data.world_110m.url, "countries")
CORES = ["#2a78d6", "#1baf7a", "#eda100", "#eb6834"]
NOMES = ["Baixa fecundidade", "Transição avançada", "Transição inicial", "Alta fecundidade"]

grupos = pd.read_csv(RAIZ / "data" / "processed" / "grupos_por_epoca.csv")
EPOCAS = sorted(grupos["epoca"].unique())
print("Épocas:", EPOCAS)

# --- Figuras: fundo branco, texto preto, salvas em figures/ ---
import re as _re
import vl_convert as vlc
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_FIG = RAIZ / "figures"
PASTA_FIG.mkdir(exist_ok=True)
PREFIXO_FIG = "07"
_renderer_padrao = alt.renderers.get()
_contador_fig = {"n": 0}

def _titulo_de(spec):
    t = spec.get("title")
    if isinstance(t, dict):
        t = t.get("text")
    if isinstance(t, list):
        t = " ".join(map(str, t))
    if isinstance(t, str) and t:
        return t
    for chave in ("layer", "hconcat", "vconcat", "concat"):
        for sub in spec.get(chave, []):
            r = _titulo_de(sub)
            if r:
                return r
    return None

def _nome_grafico(spec):
    t = _titulo_de(spec) or "chart"
    slug = _re.sub(r"[^a-zA-Z0-9]+", "_", str(t).lower()).strip("_")
    return slug[:60] or "chart"

def _renderer_com_figuras(spec, **metadata):
    _contador_fig["n"] += 1
    base = f"{PREFIXO_FIG}_{_contador_fig['n']:02d}_{_nome_grafico(spec)}"
    spec_branco = dict(spec, background="white")
    (PASTA_FIG / f"{base}.png").write_bytes(vlc.vegalite_to_png(spec_branco, scale=2))
    if '"params"' in str(spec) and '"bind"' in str(spec):
        alt.VegaLite(spec_branco).save(str(PASTA_FIG / f"{base}.html"))
    return _renderer_padrao(spec, **metadata)

alt.renderers.register("com_figuras", _renderer_com_figuras)
alt.renderers.enable("com_figuras")

EN_GRUPO = {
    "Baixa fecundidade": "Low fertility",
    "Transição avançada": "Advanced transition",
    "Transição inicial": "Early transition",
    "Transição intermediária": "Intermediate transition",
    "Alta fecundidade": "High fertility",
}


Épocas: [np.int64(1950), np.int64(1970), np.int64(1990), np.int64(2010), np.int64(2030), np.int64(2050), np.int64(2100)]


Valores baixos indicam que as covariáveis explicam pouco da variação. Valores muito altos, com razões de chances extremas, podem indicar separação quase perfeita, que é o caso desta versão.

## 1. Painel de transições

Cruzamos cada época com a seguinte. Para cada intervalo, guardamos o grupo de partida, o grupo de chegada e a fecundidade e expectativa de vida de partida.

In [2]:
valores = paises_medium[paises_medium["Time"].isin(EPOCAS)][["LocID", "Time", "TFR", "LEx"]]
grupos_valores = grupos.merge(valores, left_on=["LocID", "epoca"], right_on=["LocID", "Time"]).drop(columns="Time")

partes = []
for e0, e1 in zip(EPOCAS[:-1], EPOCAS[1:]):
    a = grupos_valores[grupos_valores["epoca"] == e0].set_index("LocID")
    b = grupos_valores[grupos_valores["epoca"] == e1].set_index("LocID")["grupo"].rename("grupo_final")
    j = a.join(b, how="inner").reset_index()
    j["intervalo"] = f"{e0}→{e1}"
    j["anos"] = e1 - e0
    partes.append(j)
painel = pd.concat(partes, ignore_index=True)

# Desfechos: avançar para grupo mais baixo; qualquer mudança de grupo
painel["avancou"] = (painel["grupo_final"] < painel["grupo"]).astype(int)
painel["mudou"] = (painel["grupo_final"] != painel["grupo"]).astype(int)

matriz_transicao = pd.crosstab(
    painel.loc[painel["anos"] == 20, "grupo"],
    painel.loc[painel["anos"] == 20, "grupo_final"],
    normalize="index",
).round(3)
matriz_transicao.index = [f"{g} {NOMES[g]}" for g in matriz_transicao.index]
matriz_transicao.columns = [f"{g} {NOMES[g]}" for g in matriz_transicao.columns]
print("Intervalos de 20 anos:", (painel["anos"] == 20).sum())
matriz_transicao

Intervalos de 20 anos: 1185


,0 Baixa fecundidade,1 Transição avançada,2 Transição inicial,3 Alta fecundidade
0 Baixa fecundidade,1.000,0.000,0.000,0.000
1 Transição avançada,0.311,0.689,0.000,0.000
2 Transição inicial,0.000,0.592,0.408,0.000
3 Alta fecundidade,0.000,0.008,0.555,0.437


Nesta versão não há retrocessos: nenhum país passa para um grupo de número maior.

## 2. Modelo logístico

O modelo usa só os países que ainda podem avançar (grupo de partida diferente de 0) e intervalos de 20 anos, que são os de comprimento uniforme. As covariáveis são padronizadas.

In [3]:
modelo_df = painel[(painel["anos"] == 20) & (painel["grupo"] > 0)].copy()
medias = modelo_df[["TFR", "LEx"]].mean()
desvios = modelo_df[["TFR", "LEx"]].std()
for c in ["TFR", "LEx"]:
    modelo_df[f"{c}_z"] = (modelo_df[c] - medias[c]) / desvios[c]

ajuste_avanco = logit("avancou ~ TFR_z + LEx_z + C(grupo)", data=modelo_df).fit(disp=0)
ajuste_mudanca = logit("mudou ~ TFR_z + LEx_z + C(grupo)", data=modelo_df).fit(disp=0)

razoes = pd.DataFrame({
    "razão de chances (avançar)": np.exp(ajuste_avanco.params),
    "p (avançar)": ajuste_avanco.pvalues,
}).round(3)
print(f"Observações: {len(modelo_df)} | avanços observados: {modelo_df['avancou'].sum()} | pseudo-R²: {ajuste_avanco.prsquared:.3f}")
razoes

Observações: 1015 | avanços observados: 469 | pseudo-R²: 0.426


,razão de chances (avançar),p (avançar)
Intercept,1.000000e-03,0.0
C(grupo)[T.2],1.488784e+04,0.0
C(grupo)[T.3],2.909889e+07,0.0
TFR_z,3.800000e-02,0.0
LEx_z,3.779000e+01,0.0


**Leitura: o modelo está instável nesta versão e não deve ser apresentado como resultado.** Em 2030 o grupo de alta fecundidade ficou vazio, e a separação entre grupos é quase perfeita: as razões de chances saem da ordem de 10⁴ a 10⁷ e todos os p-valores são zero. O pseudo-R² de 0,43 parece bom, mas é medido sobre o próprio ajuste, não sobre previsões. A causa provável é que os grupos foram definidos com fecundidade e expectativa de vida, que também entram como covariáveis: o modelo aprende a regra de classificação em vez de estimar transições. A estimativa do modelo logístico deve ser refeita (por exemplo, com regularização ou com covariáveis que não definem os grupos) antes de qualquer conclusão.

## 3. Risco de cada país

O risco é a probabilidade prevista de avanço no intervalo **2030→2050**, a partir do grupo e dos indicadores de 2030. Como 2030 é uma projeção, o risco é prospectivo. Países no grupo de baixa fecundidade não podem avançar e ficam sem valor.

Também mostramos a previsão do modelo para o intervalo **2010→2030**, que já foi observado, como verificação: a previsão deve acompanhar o que aconteceu.

In [4]:
def preparar_novos(df):
    novo = df.copy()
    novo["TFR_z"] = (novo["TFR"] - medias["TFR"]) / desvios["TFR"]
    novo["LEx_z"] = (novo["LEx"] - medias["LEx"]) / desvios["LEx"]
    return novo

atual = grupos_valores[grupos_valores["epoca"] == 2030][["LocID", "Location", "grupo", "nome", "TFR", "LEx"]].copy()
atual = preparar_novos(atual)
pode = atual["grupo"] > 0
atual["risco_avancar_2030_2050"] = np.nan
atual.loc[pode, "risco_avancar_2030_2050"] = ajuste_avanco.predict(atual.loc[pode])
atual["risco_mudar_2030_2050"] = np.nan
atual.loc[pode, "risco_mudar_2030_2050"] = ajuste_mudanca.predict(atual.loc[pode])

# Verificação: previsão para 2010→2030, já observado
verif = grupos_valores[grupos_valores["epoca"] == 2010].merge(
    painel[painel["intervalo"] == "2010→2030"][["LocID", "avancou"]], on="LocID")
verif = preparar_novos(verif)
verif = verif[verif["grupo"] > 0].copy()
verif["previsto"] = ajuste_avanco.predict(verif)
print("Verificação 2010→2030: média prevista =", round(verif["previsto"].mean(), 3),
      "| média observada =", round(verif["avancou"].mean(), 3))

Verificação 2010→2030: média prevista = 0.445 | média observada = 0.453


**Leitura da verificação:** a previsão média para 2010→2030 (0,445) coincide com a observada (0,453), mas essa comparação é feita dentro da própria amostra e não valida o modelo para o futuro. Com separação quase perfeita, esse ajuste é esperado e não mostra capacidade de previsão.

## 4. Mapa do risco

Cada país é colorido pelo risco de avançar entre 2030 e 2050. Países em cinza estão no grupo de baixa fecundidade (não podem avançar) ou não estão na malha.

In [5]:
escala_risco = alt.Scale(domain=[0, 1], range=["#fcfcfb", "#6fa6e8", "#0b3a7a"], clamp=True)
mapa_risco = (
    alt.Chart(MALHA)
    .mark_geoshape(stroke="white", strokeWidth=0.3)
    .transform_filter("datum.id != 10")
    .transform_lookup(lookup="id", from_=alt.LookupData(atual.assign(nome=atual["nome"].map(EN_GRUPO)), "LocID", ["risco_avancar_2030_2050", "Location", "nome"]))
    .encode(
        color=alt.condition(
            "isValid(datum.risco_avancar_2030_2050)",
            alt.Color("risco_avancar_2030_2050:Q", scale=escala_risco,
                      legend=alt.Legend(title="Probability of advancing")),
            alt.value("#d9d9d9"),
        ),
        tooltip=[alt.Tooltip("Location:N", title="Country"), alt.Tooltip("nome:N", title="Group in 2030"),
                 alt.Tooltip("risco_avancar_2030_2050:Q", title="Risk", format=".2f")],
    )
    .project(type="equalEarth")
    .properties(width=680, height=340, title="Risk of advancing group, 2030→2050")
)
mapa_risco

alt.Chart(...)

**Leitura:** em 2030 o grupo de alta fecundidade está vazio, então o mapa tem só os grupos de transição avançada e inicial. Os riscos vão de cerca de 0,01 a 0,99, e a maioria dos países de transição inicial fica acima de 0,9. Esses valores vêm do modelo instável descrito acima e devem ser lidos como ilustração, não como previsão. Para diferenciar países dentro de um grupo, o modelo precisaria de covariáveis que não definem os grupos, como mortalidade infantil, migração ou região.

## 5. Países com maior risco

Os 15 países com maior probabilidade de avanço, entre os que não estão no grupo de baixa fecundidade.

In [6]:
top = (
    atual.dropna(subset=["risco_avancar_2030_2050"])
    .sort_values("risco_avancar_2030_2050", ascending=False)
    .head(15)[["Location", "nome", "TFR", "risco_avancar_2030_2050", "risco_mudar_2030_2050"]]
    .rename(columns={"Location": "país", "nome": "grupo em 2030", "TFR": "TFR 2030",
                     "risco_avancar_2030_2050": "risco de avançar", "risco_mudar_2030_2050": "risco de mudar"})
)
top.round(3)

,país,grupo em 2030,TFR 2030,risco de avançar,risco de mudar
760,Yemen,Transição inicial,3.971,0.994,0.994
130,Zambia,Transição inicial,3.666,0.992,0.992
340,Guinea-Bissau,Transição inicial,3.370,0.990,0.990
361,Mauritania,Transição inicial,4.237,0.989,0.989
172,Congo,Transição inicial,3.780,0.989,0.989
137,Zimbabwe,Transição inicial,3.378,0.989,0.989
235,Sudan,Transição inicial,3.882,0.988,0.988
46,Madagascar,Transição inicial,3.556,0.986,0.986
123,United Republic of Tanzania,Transição inicial,4.161,0.985,0.985
396,Sierra Leone,Transição inicial,3.235,0.985,0.985


## 6. Salvar resultados

- `data/processed/risco_progressao.csv`: risco de avançar e de mudar de grupo para cada país em 2030.
- `data/processed/matriz_transicao.csv`: matriz de transição entre grupos em intervalos de 20 anos.

In [7]:
SAIDA = RAIZ / "data" / "processed"
atual[["LocID", "Location", "grupo", "nome", "TFR", "LEx", "risco_avancar_2030_2050", "risco_mudar_2030_2050"]].round(4).to_csv(
    SAIDA / "risco_progressao.csv", index=False)
matriz_transicao.to_csv(SAIDA / "matriz_transicao.csv")
print("Salvos em", SAIDA)

Salvos em /Users/marcosabilio/Documents/GitHub/DataQuest2026/data/processed
